[Lab README](README.md)

# Lab 2.2: Full-text retrieval, with and without the graph

Embeddings blur exact strings. A postal code, a confirmation number, and a
product SKU all embed to roughly "short numeric token", which is why a question
containing one so often returns chunks about the right topic and the wrong
record.

Full-text search does not have that problem, and it has the opposite one: it
cannot match a paraphrase. Hybrid retrieval runs both and fuses the rankings.

There are two comparisons below, in order.

First, `VectorRetriever` against `HybridRetriever`, on the same question about a
postal code. The full-text arm has to earn its place on an exact identifier that
carries almost no semantic signal, so the delta is visible as a rank rather than
as an argument.

Then `HybridCypherRetriever`, which is that same fusion plus a traversal. That
one is the production shape: Lab 3 puts it behind a tool, Lab 4 writes against
the `hotel_id` it returns, and Lab 5 deploys it unchanged.

In [ ]:
# At an AWS event the dependencies are already installed. Self-paced, run
# `uv venv && uv pip install -r requirements.txt` inside 02-retrieval first,
# then run this cell to confirm the kernel can see what the notebook imports.
import importlib.util
import sys

REQUIRED = ("boto3", "dotenv", "neo4j", "neo4j_graphrag", "strands", "workshop")
absent = [name for name in REQUIRED if importlib.util.find_spec(name) is None]
if absent:
    raise ModuleNotFoundError(
        f"Not importable: {', '.join(absent)}. Install this lab's "
        "requirements.txt, then restart the kernel."
    )

print(f"Python {sys.version_info.major}.{sys.version_info.minor}")
print(f"Imports resolve: {', '.join(REQUIRED)}")

## Connect and verify the graph

Retrieval notebooks do not create schema artifacts. This cell requires both Lab 1
indexes to be online with the expected label, property, dimensions, and
similarity function, then checks the graph facts the questions below depend on.

In [ ]:
import os

import boto3
from dotenv import load_dotenv
from IPython.display import HTML, display
from neo4j import GraphDatabase

load_dotenv()

NEO4J_VARS = ("NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD")
missing = [name for name in NEO4J_VARS if not os.environ.get(name)]
has_aws = boto3.Session().get_credentials() is not None
RETRIEVAL_READY = not missing and has_aws

if missing:
    print(f"Neo4j is not configured: set {', '.join(missing)} in the repo-root .env")
if not has_aws:
    print("No AWS credentials found, so the Bedrock calls below cannot run")

if RETRIEVAL_READY:
    # Imported here because `workshop.graph_connection` raises at import when
    # NEO4J_PASSWORD is unset, which would fail this cell instead of skipping it.
    from workshop.graph_connection import NEO4J_URI, neo4j_auth
    from workshop.retrieval_contract import (
        CHUNK_FULLTEXT_INDEX,
        CHUNK_VECTOR_INDEX,
        EMBEDDING_DIMENSIONS,
        EMBEDDING_MODEL_ID,
    )
    from workshop.retrieval_setup import fixture_problems, verify_retrieval_indexes

    driver = GraphDatabase.driver(NEO4J_URI, auth=neo4j_auth())
    driver.verify_connectivity()

    try:
        verify_retrieval_indexes(driver)
        problems = fixture_problems(driver)
        if problems:
            raise RuntimeError("; ".join(problems))
    except Exception as exc:
        raise RuntimeError(
            f"The graph is not ready for retrieval: {exc}\n"
            "Run Lab 1 (01-graph-build/1.1_build_graph.ipynb) first."
        ) from exc

    print(f"{CHUNK_VECTOR_INDEX} is ONLINE")
    print(f"{CHUNK_FULLTEXT_INDEX} is ONLINE")
    print("Every graph fact these questions depend on is present")
else:
    print("\nThe cells below will skip. Finish Lab 0 and Lab 1, then come back.")

## The schema the graph actually holds

Lab 1 pinned this schema during extraction, so the traversals below can name
relationships instead of discovering them. Render it before using any retriever
that traverses, so the shape of the added context is predictable.

In [ ]:
from workshop.graph_schema import GRAPH_SCHEMA

pattern_rows = "".join(
    f"<tr><td><strong>{source}</strong></td>"
    f"<td><code>-[:{relationship}]-&gt;</code></td>"
    f"<td><strong>{target}</strong></td></tr>"
    for source, relationship, target in GRAPH_SCHEMA["patterns"]
)
display(HTML(
    "<table><thead><tr><th>From</th><th>Relationship</th><th>To</th></tr>"
    f"</thead><tbody>{pattern_rows}</tbody></table>"
    "<p>Each extracted entity also points to its source "
    "<code>(entity)-[:FROM_CHUNK]-&gt;(:Chunk)</code>.</p>"
))

## One embedder, shared with the build

A query embedding has to come from the same model, at the same dimension, with
the same purpose as the vectors Lab 1 wrote. A mismatch does not raise. It
returns confident, wrong neighbours. So the embedder is constructed from the
same module the build used rather than configured again here.

In [ ]:
if not RETRIEVAL_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    from neo4j_graphrag.types import RetrieverResultItem

    from workshop.bedrock_providers import BedrockEmbeddings

    embedder = BedrockEmbeddings(region_name=os.environ.get("AWS_REGION", "us-east-1"))
    print(f"model: {EMBEDDING_MODEL_ID}")
    print(f"dimensions: {EMBEDDING_DIMENSIONS}")

    def chunk_text_formatter(record):
        """Return the chunk text itself.

        The library default is `content=str(node)`, which prints the whole node
        map with its newlines escaped and is unreadable on a projector.
        """
        node = record.get("node") or {}
        return RetrieverResultItem(
            content=node.get("text") or "",
            metadata={"score": record.get("score")},
        )

    def show_results(question, result, why):
        """Print each retrieved item with its score, then why the pattern fits."""
        print(f"Question: {question}\n")
        for number, item in enumerate(result.items, 1):
            score = (item.metadata or {}).get("score")
            score_text = "n/a" if score is None else f"{score:.4f}"
            content = str(item.content)
            preview = content[:700] + ("…" if len(content) > 700 else "")
            print(f"[{number}] score={score_text}\n{preview}\n")
        print(f"Why this fits: {why}")

## Pattern 1: hybrid retrieval for an exact identifier

The chunk for Windward Mile Tower contains the postal code `60611`. Digits carry
almost no semantic signal, so a vector-only retriever ranks that chunk on the
policy wording around the code rather than on the code itself.

The cell below runs both arms of the comparison and prints the settings it used,
because they are tuned rather than default:

- The vector arm gets the whole question, embedded once.
- The full-text arm gets the bare string `60611`, so Lucene has an exact term to
  match instead of a sentence.
- Fusion is `ranker="linear", alpha=0.2`, which weights the full-text arm at
  0.8. The library default is `ranker="naive"`. This setting is the one being
  demonstrated, and the production retriever in Pattern 2 takes neither of them
  from the caller.

Both result sets print in full, followed by the line worth reading: the rank of
the `60611` chunk in each. Two assertions follow, one per arm, and they are the
honest part of the comparison.

In [ ]:
if not RETRIEVAL_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    from neo4j_graphrag.retrievers import HybridRetriever, VectorRetriever

    # Named once so the printed disclosure and the call cannot drift apart.
    TOP_K = 5
    RANKER = "linear"
    ALPHA = 0.2
    FULLTEXT_TERM = "60611"

    vector_retriever = VectorRetriever(
        driver=driver,
        index_name=CHUNK_VECTOR_INDEX,
        embedder=embedder,
        return_properties=["text"],
        result_formatter=chunk_text_formatter,
    )
    identifier_question = "What is the cancellation policy for the hotel at 60611?"
    vector_identifier_result = vector_retriever.search(
        query_text=identifier_question,
        top_k=TOP_K,
    )

    hybrid_retriever = HybridRetriever(
        driver=driver,
        vector_index_name=CHUNK_VECTOR_INDEX,
        fulltext_index_name=CHUNK_FULLTEXT_INDEX,
        embedder=embedder,
        return_properties=["text"],
        result_formatter=chunk_text_formatter,
    )
    hybrid_result = hybrid_retriever.search(
        query_text=FULLTEXT_TERM,
        query_vector=vector_identifier_result.metadata["query_vector"],
        top_k=TOP_K,
        ranker=RANKER,
        alpha=ALPHA,
    )

    print(
        f"Settings: top_k={TOP_K}, ranker={RANKER!r}, alpha={ALPHA}, so the "
        f"full-text arm carries {1 - ALPHA:.1f} of the fused score\n"
    )
    show_results(
        identifier_question,
        vector_identifier_result,
        "This is the semantic-only comparison. An exact postal code is not a strong semantic signal.",
    )
    print("\n" + "=" * 80 + "\n")
    show_results(
        f"{identifier_question}   [vector arm: the question above; "
        f"full-text arm: {FULLTEXT_TERM!r}]",
        hybrid_result,
        "Full-text matching preserves 60611 while the separately supplied question vector handles the policy wording.",
    )

    def rank_of(result, term):
        """Return the 1-based rank of the first item containing term, or None."""
        for number, item in enumerate(result.items, 1):
            if term in str(item.content):
                return number
        return None

    vector_rank = rank_of(vector_identifier_result, FULLTEXT_TERM)
    hybrid_rank = rank_of(hybrid_result, FULLTEXT_TERM)
    print("\n" + "=" * 80)
    print(
        f"Rank of the {FULLTEXT_TERM} chunk out of {TOP_K}: "
        f"vector={vector_rank or 'not returned'}, hybrid={hybrid_rank or 'not returned'}"
    )

    # Both arms are checked, so the printed verdict cannot contradict the prose
    # above it on a run where the vector arm does better than expected.
    assert hybrid_rank is not None, "the 60611 chunk is missing from the hybrid results"
    assert vector_rank is None or hybrid_rank <= vector_rank, (
        "the vector-only arm ranked the 60611 chunk above the hybrid arm, "
        "so fusion is not earning its place on this question"
    )

    if vector_rank is None:
        print("The vector arm did not return it at all. The full-text arm found it.")
    elif hybrid_rank < vector_rank:
        print(f"Fusion moved it from rank {vector_rank} to rank {hybrid_rank}.")
    else:
        print("Both arms returned it at the same rank on this run.")

## Pattern 2: the production retriever

`HybridCypherRetriever` is the fusion you just ran plus a traversal in the same
shape as the one in 2.1, deliberately narrower. It returns `hotel_id`,
`hotel_name`, `address`, `guest_rating`, and up to twelve amenity names sorted
alphabetically. No rooms, no policies, no services. A tool contract is a promise
about what comes back, and the narrow promise is the one that stays true when a
model, rather than you, is reading the result.

It is also the one retriever the rest of this workshop uses, so it is not built
inline here. It lives in `workshop.hybrid_retrieval`, and the tool that wraps it
accepts exactly one field:

```python
search_hotel_knowledge(query: str) -> list[HotelEvidence]
```

No ranker, no alpha, no top-k, no retriever-mode selector. Those are the
comparisons you just ran; a caller does not get to re-run them at request time.
Fusion is `NAIVE`, `top_k` is fixed, and the traversal is the same reviewed
Cypher every time.

That narrowness is the point. Lab 3 hands this function to an agent, and an
agent that cannot choose a ranker cannot choose a bad one.

### The hero question

> **What amenities and guest rating does AnyCompany Cairo Nile View have?**

Every arm of the retriever earns its place on this one question. The exact hotel
name is what the full-text arm is for. "Amenities and guest rating" is a
paraphrase of how the document actually words it, which is what the vector arm
is for.

Both arms return the same thing: chunk text. The rating and the amenities are in
that text, as prose an answering model has to re-read and re-extract, and the
cell below prints enough of it to see `**Guest Rating:** 4.5/5.0` sitting a few
lines in. The traversal returns those same facts as named fields instead:
`guest_rating` comes back as the number `4.5`, and `amenities` comes back as a
list built from the hotel's `OFFERS_AMENITY` edges rather than from whatever the
matched chunk happens to mention. A field is queryable, assertable, and safe to
hand to a write path. A sentence is none of those.

The cell prints the fusion's own output first and the traversal's second, so the
difference is watched rather than described.

In [ ]:
if not RETRIEVAL_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    from workshop.contracts import HYBRID_RANKER, HYBRID_TOP_K
    from workshop.graph_setup import HERO_NAME
    from workshop.hybrid_retrieval import search_hotel_knowledge

    hero_question = f"What amenities and guest rating does {HERO_NAME} have?"

    # Left side: the fusion on its own, at the library's default ranker, with no
    # traversal behind it. All it can return is text.
    fusion_only = hybrid_retriever.search(query_text=hero_question, top_k=1)
    print(f"Question: {hero_question}\n")
    print("Fusion alone, no traversal. The top result, as text:")
    print(str(fusion_only.items[0].content)[:600] if fusion_only.items else "  nothing returned")

    # Right side: the same fusion plus the reviewed traversal.
    results = search_hotel_knowledge(hero_question)
    if not results:
        raise RuntimeError(
            "The production retriever returned nothing for the hero question. "
            "The graph is missing the fixture hotel: re-run Lab 1 and let it finish."
        )
    top = results[0]

    print("\n" + "=" * 80 + "\n")
    print(f"Fusion plus traversal. ranker={HYBRID_RANKER}  top_k={HYBRID_TOP_K}  traversal=fixed\n")
    print(f"Hotel: {top['hotel_name']}  (hotel_id={top['hotel_id']})")
    print(f"Combined hybrid score: {top['combined_score']:.4f}")
    print(f"Guest rating: {top['guest_rating']}")
    print(f"Exact matched terms: {', '.join(top['exact_terms']) or 'none'}")
    print(f"Amenities ({len(top['amenities'])}): {', '.join(top['amenities'])}")
    print("\nChunk evidence, carried alongside those fields:")
    print(top["chunk_evidence"][:600])

### What just happened

- **Vector matching** handled the paraphrased request for "amenities and guest rating".
- **Full-text matching** locked onto the exact hotel name and location terms.
- The **reviewed Cypher traversal** followed the matched chunk to its hotel and
  returned up to 12 connected amenities, the guest rating, and the opaque
  `hotel_id`. That `hotel_id`, never the display name, is the identity the
  reservation command accepts in Lab 4.

## A question the evidence cannot answer

> **Does AnyCompany Cairo Nile View guarantee room availability next weekend?**

The graph holds hotel knowledge, not live inventory. Below, the same one
retrieval tool goes to a small local agent carrying the workshop's grounding
instructions, and the agent answers the hero question first.

Then three cells, in order: the grounding instructions themselves, which are the
lever; the agent answering the availability question; and the tool result the
agent actually received, recorded as it was handed over. A refusal is only worth
anything if you can see what the model had to work with, and the payload worth
showing is the agent's own rather than a matching call made beside it.

In [ ]:
if not RETRIEVAL_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    import json

    from strands import Agent, tool
    from strands.models import BedrockModel

    from workshop.bedrock_providers import default_model_id
    from workshop.hybrid_retrieval import GROUNDING_INSTRUCTIONS

    AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
    # One definition, in workshop/src/workshop/bedrock_providers.py.
    # A MODEL_ID in the environment still overrides it.
    MODEL_ID = default_model_id()

    # Every call the agent makes is recorded here, so the evidence shown below
    # is the payload the model actually received rather than a second lookup.
    TOOL_CALLS = []

    @tool
    def search_hotel_knowledge_tool(query: str) -> str:
        """Search grounded hotel evidence and return bounded JSON facts."""
        evidence = search_hotel_knowledge(query)
        TOOL_CALLS.append({"query": query, "evidence": evidence})
        return json.dumps(evidence, ensure_ascii=False)

    grounded_agent = Agent(
        model=BedrockModel(model_id=MODEL_ID, region_name=AWS_REGION),
        tools=[search_hotel_knowledge_tool],
        system_prompt=(
            "You are a grounded hotel-information assistant. Call "
            "search_hotel_knowledge_tool before answering any hotel question.\n\n"
            + GROUNDING_INSTRUCTIONS
        ),
    )
    # Strands prints the streamed response itself, through the default
    # PrintingCallbackHandler. Wrapping this call in print() would show the
    # whole answer a second time.
    grounded_agent(hero_question)

In [ ]:
if not RETRIEVAL_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    availability_question = (
        f"Does {HERO_NAME} guarantee room availability next weekend?"
    )

    print("The grounding instructions the agent is carrying:\n")
    print(GROUNDING_INSTRUCTIONS)
    print("\n" + "=" * 80)
    print(f"\nThe question going to the agent next: {availability_question}")

In [ ]:
if not RETRIEVAL_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    calls_before = len(TOOL_CALLS)
    # As above, Strands prints the streamed response; no print() wrapper here.
    grounded_agent(availability_question)

In [ ]:
if not RETRIEVAL_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    agent_calls = TOOL_CALLS[calls_before:]
    if agent_calls:
        print(f"Retriever calls the agent made on that question: {len(agent_calls)}")
        print(f"Its last query was: {agent_calls[-1]['query']}\n")
        availability_evidence = agent_calls[-1]["evidence"]
    else:
        print(
            "The agent answered without calling the retriever, so the same "
            "question goes to the retriever directly.\n"
        )
        availability_evidence = search_hotel_knowledge(availability_question)

    print("What the agent had to work with, field by field:\n")
    for number, item in enumerate(availability_evidence, 1):
        print(f"[{number}] hotel_name:     {item['hotel_name']}")
        print(f"    hotel_id:       {item['hotel_id']}")
        print(f"    address:        {item['address']}")
        print(f"    guest_rating:   {item['guest_rating']}")
        print(f"    amenities:      {', '.join(item['amenities']) or 'none'}")
        print(f"    exact_terms:    {', '.join(item['exact_terms']) or 'none'}")
        print(f"    chunk_evidence: {item['chunk_evidence'][:200]}…\n")

    # The frozen evidence contract has no field for inventory on a date, so this
    # check holds without depending on what the model happened to say above.
    AVAILABILITY_FIELDS = ("availability", "available_rooms", "vacancies", "dates")
    assert not any(
        field in item
        for item in availability_evidence
        for field in AVAILABILITY_FIELDS
    ), "the evidence contract grew an availability field, so this close no longer holds"

    fields = sorted(availability_evidence[0]) if availability_evidence else []
    print(f"Fields returned per result: {', '.join(fields) or 'nothing was returned'}")
    print("Not one of them is availability on a date.")

The evidence above is the whole story, and it is the agent's own tool result
rather than a second lookup. The retriever returned hotel fields and chunk text
and nothing about inventory on a date, the grounding instructions say that
"subject to availability" is a policy rather than a vacancy, and the agent
declined instead of reading one as the other.

Note for facilitators: the refusal is a live model call, so its wording changes
from run to run. The assertion in the cell above deliberately covers the
evidence rather than the prose, because the evidence is what the contract fixes.
Read the answer for whether it declines, not for a particular sentence.

## The mechanism, in one sentence

Vector search finds candidates, traversal finds what is connected.

**Next:** [Lab 3: Agents and tools](../03-agents-and-tools/) gives this same
`search_hotel_knowledge` function, unchanged, to an agent named `hotel_agent`,
alongside lifecycle hooks and a booking tool. The `@tool` wrapper above is the
starting point rather than the new material. `2.3_text2cypher.ipynb` is optional
and covers the one question shape none of these retrievers handles well:
counting.

In [ ]:
if RETRIEVAL_READY:
    driver.close()
    print("This notebook's driver is closed.")
    print(
        "The driver inside workshop.hybrid_retrieval is cached and stays open "
        "for the rest of this kernel session, which is what makes the tool warm."
    )